In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xmf6
from src_gypsum import gwf, gwt, plot, utils
linea = 50*chr(0x2015)

In [2]:
#
# Variables de entorno
# -----
with open('../env.json', 'r', encoding='utf-8') as file:
    env = json.load(file)
env["ROOT_DIR"] = os.getcwd() # Agregamos el dir raíz

xmf6.nice_print("Environment variables", env)

#
# DEFINICIÓN DE LAS RUTAS Y NOMBRES
# -----
paths = dict(
    #
    # Ejecutable de Modflow: WINDOWS
    # -----
    mf6_exe = env["MF6EXE"],
    
    #
    # Directorio de trabajo para WMA
    # -----
    rt_working_dir = os.path.join(env["ROOT_DIR"], "rt_workingDir"),
    
    #
    # Nombre de los modelos de flujo y transp y espacios de trabajo
    # -----
    flow_name = "flow",
    flow_ws = "io_mf6/gwf_gwt_comp", 
    tran_name = "transport",
    tran_ws = "io_mf6/gwf_gwt_comp",
    
    #
    # Ejecutable REMIX y archivo de entrada
    # -----
    remix_exe = os.path.join(env["RT_BINARIES"], "interfaz_remix.exe"),
    remix_input = os.path.join(env["ROOT_DIR"], "datos_interfaz_gypsum.txt"),
)
#
# Nombre de los archivos de datos
# -----
paths["u_init"] = os.path.join(paths["rt_working_dir"], "u_init_yeso.dat")
paths["u_tilde"] = os.path.join(paths["rt_working_dir"], "u_tilde_yeso.dat")
paths["u_new"] = os.path.join(paths["rt_working_dir"], "u_new_yeso.out")
#
# Comando para ejeuctar interfaz_remix.exe
# -----
paths["command"] = f"{paths['remix_exe']} < {paths['remix_input']}"

xmf6.nice_print("Paths and filenames", paths)


Environment variables
―――――――――――――――――――――
     MF6EXE = C:/Users/luiggi/Documents/GitSites/mf6_tutorial/mf6/windows/mf6
     MF6DLL = C:/Users/luiggi/Documents/GitSites/mf6_tutorial/mf6/windows/libmf6.dll
RT_BINARIES = C:/Users/luiggi/Documents/GitSites/RTWMA/benchmarks/00_RT_EXE/bin
   ROOT_DIR = C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D
―――――――――――――――――――――

Paths and filenames
―――――――――――――――――――
       mf6_exe = C:/Users/luiggi/Documents/GitSites/mf6_tutorial/mf6/windows/mf6
rt_working_dir = C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D\rt_workingDir
     flow_name = flow
       flow_ws = io_mf6/gwf_gwt_comp
     tran_name = transport
       tran_ws = io_mf6/gwf_gwt_comp
     remix_exe = C:/Users/luiggi/Documents/GitSites/RTWMA/benchmarks/00_RT_EXE/bin\interfaz_remix.exe
   remix_input = C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D\datos_interfaz_gypsum.txt
        u_init = C:\Users\luiggi\Docume

In [3]:
#
#
# --- DATOS PARA LA SIMULACIÓN ---
#
print(linea)
print("Generating initial conditions".center(50))
components, especies = utils.initial_conditions(paths, comp = True, spec=True)
ncomp = len(components["names"]) # "ca+2 - so4-2", "cl-"
print(ncomp)
nspec = ncomp + 1 # ca+2, cl-, so4-2
print(linea)
#
# Discretización del espacio
nlay, nrow, ncol = 1, 1, 30
delr, delc, delz = 1, 1, 1  #0.1, 1, 1 (dx,dy,dz)
dis = {
    'length_units' : "meters",
    'nlay': nlay, 
    'nrow': nrow, 
    'ncol': ncol,
    'delr': delr, 
    'delc': delc, 
    'top' : delz, 
    'botm': 0.0 
}
xmf6.nice_print("Spatial discretization", dis)
#
# Discretización del tiempo
#
# Paso del tiempo
dt = 1.0e0
NT = 40 #repeticiones del ciclo externo
tdis = {
    'units': "days",
    'nper' : 1,
    'perioddata': [(dt, 1, 1.0)]
}
xmf6.nice_print("Time discretization", tdis)
#
# Parámetros físicos.
phys = dict(
    initial_head = 1.0,
    bc_head_t1 = [("CHD-1" , [(0, 0, dis['ncol'] - 1), 1.0])],
    porosity = 0.5,  # Porosity of mobile domain (unitless)
    hydraulic_conductivity = 1.0,  # Hydraulic conductivity ($cm s^{-1}$)
    specific_discharge = 0.2,  # inflow ($m d^{-1}$)
    longitudinal_dispersivity = 0.5, # 
    components = components["names"], # Components names
    source_concentration = components["ext_water"],  # Source concentration (unitless)
    initial_concentration = components["init_conc"], # Initial concentration (unitless)
)
# Agregamos la información del pozo
q = phys["specific_discharge"] * dis['delc'] * (dis['top'] - dis['botm'])
for m in range(0, ncomp):
    phys[f"well_c{m+1}"] = [("WEL-1", "AUX", "CONCENTRATION"), ((0, 0, 0), q, phys["source_concentration"][m])]

xmf6.nice_print("Physical parameters", phys)

――――――――――――――――――――――――――――――――――――――――――――――――――
          Generating initial conditions           
-> File C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D\rt_workingDir\u_init_yeso.dat already exist
2
――――――――――――――――――――――――――――――――――――――――――――――――――

Spatial discretization
――――――――――――――――――――――
length_units = meters
        nlay = 1
        nrow = 1
        ncol = 30
        delr = 1
        delc = 1
         top = 1
        botm = 0.0
――――――――――――――――――――――

Time discretization
―――――――――――――――――――
     units = days
      nper = 1
perioddata = ―― data array ――
           = (1.0, 1, 1.0)
―――――――――――――――――――

Physical parameters
―――――――――――――――――――
             initial_head = 1.0
               bc_head_t1 = ―― data array ――
                          = ('CHD-1', [(0, 0, 29), 1.0])
                 porosity = 0.5
   hydraulic_conductivity = 1.0
       specific_discharge = 0.2
longitudinal_dispersivity = 0.5
               components = ―― data array ――
        

In [4]:
phys["initial_concentration"]

[array([  1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,   1., 200.,
          1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,
          1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.]),
 array([  2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2., 400.,
          2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,
          2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.])]

In [5]:
especies["init_conc"]

[array([  1.00003,   1.00003,   1.00003,   1.00003,   1.00003,   1.00003,
          1.00003,   1.00003,   1.00003,   1.00003, 200.     ,   1.00003,
          1.00003,   1.00003,   1.00003,   1.00003,   1.00003,   1.00003,
          1.00003,   1.00003,   1.00003,   1.00003,   1.00003,   1.00003,
          1.00003,   1.00003,   1.00003,   1.00003,   1.00003,   1.00003]),
 array([  2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2., 400.,
          2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.,
          2.,   2.,   2.,   2.,   2.,   2.,   2.,   2.]),
 array([3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05,
        3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05,
        1.64691e-07, 3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05,
        3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05,
        3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-05,
        3.29371e-05, 3.29371e-05, 3.29371e-05, 3.29371e-0

In [6]:
tiempos_plot = [10, 20, 30, 40]

u_tilde = [0 for i in range(ncomp)]
c_final = {}
c_final[0.0] = especies["init_conc"]#phys["initial_concentration"]

for it in range(1, NT + 1):
    ts = it * dt
    xmf6.nice_print(f"Step = {it}, time = {ts}d")

    # =====================================================
    # GWF + GWT para cada componente
    # =====================================================

    for m in range(ncomp):
        # -------------------------------------------------
        # GWF
        # -------------------------------------------------
#        o_sim_f = gwf.build(phys,dis,m,paths)
        o_sim_f, o_gwf = gwf.build(paths, tdis, phys, dis, m, silent = True)
#        if it in tiempos_plot:
        print(" GWF", end="")
        o_sim_f.run_simulation(silent=True)

        # -------------------------------------------------
        # GWT
        # -------------------------------------------------
#        o_sim_t = gwt.build(phys,dis,m,paths)
        o_sim_t, o_gwt = gwt.build(paths, tdis, phys, dis, m, silent = True)
#        if it in tiempos_plot:
        print(f"-GWT: C{m+1}", end=" |")
        o_sim_t.run_simulation(silent=True)

        # -------------------------------------------------
        # Concentración calculada
        # -------------------------------------------------
        u_tilde[m] = utils.get_conc(o_sim_t)

    # =====================================================
    # Guardar u_tilde_yeso.dat
    # =====================================================

    #
    # --- Guardamos concentraciones en "u_tilde.dat"
    print("\n-> Writting: u_tilde.dat")
    with open(paths["u_tilde"], "w") as f:
        for a in u_tilde:
            f.writelines(" ".join([c for c in a.astype(str)]))
            f.write("\n")

    # =====================================================
    # Ejecutar REMIX
    # =====================================================
    #
    # ---- Ejecutamos REMIX usando las nuevas concentraciones
    print(f"-> Executing: {paths["command"]}\n")
    ret = os.system(paths["command"])
    print(f"\n-> Error code: {ret}")

    # =====================================================
    # Leer u_new_yeso.out
    # =====================================================
    #
    # --- Leemos las concentraciones de las componentes calculadas por REMIX
    print(f"-> Reading: {paths["u_new"]}")
    with open(paths["u_new"], "r") as f:
        lines = f.readlines()

    # =====================================================
    # Componentes después de REMIX
    # =====================================================
    #
    # Actualizamos las concentraciones para GWT
    key_string = "Aqueous component concentrations after solving reactive mixing iteration (rows: components, columns: targets):"
    iccn = utils.find_index(lines, key_string, 2)
    u_new = [np.array(l.split(), dtype=float) for l in lines[iccn:iccn+ncomp]]
    phys["initial_concentration"] = u_new
#    print(u_new)
    # =====================================================
    # Leer especies
    # =====================================================
    # Almacenamos las especies calculadas en el diccionario s_final
    key_string = "Concentrations of variable activity species after solving reactive mixing iteration (rows: species, columns: targets):"
    isp = utils.find_index(lines, key_string, 2)
    c_final[ts] = [np.array(l.split(), dtype=float) for l in lines[isp:isp+nspec+1]]
#    print(c_final[ts])


Step = 1, time = 1.0d
―――――――――――――――――――――

 GWF-GWT: C1 | GWF-GWT: C2 |
-> Writting: u_tilde.dat
-> Executing: C:/Users/luiggi/Documents/GitSites/RTWMA/benchmarks/00_RT_EXE/bin\interfaz_remix.exe < C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D\datos_interfaz_gypsum.txt


-> Error code: 0
-> Reading: C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D\rt_workingDir\u_new_yeso.out

Step = 2, time = 2.0d
―――――――――――――――――――――

 GWF-GWT: C1 | GWF-GWT: C2 |
-> Writting: u_tilde.dat
-> Executing: C:/Users/luiggi/Documents/GitSites/RTWMA/benchmarks/00_RT_EXE/bin\interfaz_remix.exe < C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D\datos_interfaz_gypsum.txt


-> Error code: 0
-> Reading: C:\Users\luiggi\Documents\GitSites\RTWMA\benchmarks\03_RT_COM_gypsum_1D\rt_workingDir\u_new_yeso.out

Step = 3, time = 3.0d
―――――――――――――――――――――

 GWF-GWT: C1 | GWF-GWT: C2 |
-> Writting: u_tilde.dat
-> Executing: C:/Users/luiggi/Document

In [7]:
c_final.keys()

dict_keys([0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0])

In [8]:
print("\n-> Writting final results:")
with open(os.path.join(paths["rt_working_dir"], 
                       "gypsum_final_species.dat"), "w") as f:
    for k in c_final.keys():
        print(k)
        for c in range(0,3):
            l = [c for c in c_final[k][c].astype(str)]
            l.insert(0, f"{k:4.2f}")
            l.append("\n")
            f.write(" ".join(l))


-> Writting final results:
0.0
1.0
2.0
3.0
4.0
5.0
6.0
7.0
8.0
9.0
10.0
11.0
12.0
13.0
14.0
15.0
16.0
17.0
18.0
19.0
20.0
21.0
22.0
23.0
24.0
25.0
26.0
27.0
28.0
29.0
30.0
31.0
32.0
33.0
34.0
35.0
36.0
37.0
38.0
39.0
40.0
